# ⚙️ Notebook 2 — Preprocessing & Feature Engineering
**Input:** `speeches_clean.csv`  **Output:** `speeches_features.csv`

The speeches file already contains `hour_of_day`, `time_bin`, `hour_sin`, `hour_cos`,
`day_of_week`, `month`, `is_weekend`. This notebook validates those features,
adds missing ones, encodes categoricals, checks class imbalance and creates the
train/val/test split.


In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, matplotlib.ticker as mtick
import seaborn as sns, warnings, json
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi':130,'axes.spines.top':False,'axes.spines.right':False})
BLUE,RED,GREEN,ORANGE='#2B5797','#C0392B','#27AE60','#E67E22'
df = pd.read_csv('speeches_clean.csv', low_memory=False)
print(f'Loaded {len(df):,} rows, {df.shape[1]} columns')

## 1. Text Cleaning

In [ ]:
import re

def clean_speech(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'<[^>]+>', ' ', text)                                # XML/HTML tags (source is XML)
    text = re.sub(r'[^\x20-\x7E\u00C0-\u024F\u0300-\u036F]', ' ', text)  # keep Dutch diacritics
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['speech_text_clean'] = df['speech_text'].apply(clean_speech)
n_changed = (df['speech_text_clean'] != df['speech_text'].fillna('')).sum()
print(f"Speeches modified by cleaning: {n_changed:,}")
print("\nBefore:", str(df['speech_text'].iloc[0])[:200])
print("After :", df['speech_text_clean'].iloc[0][:200])

## 2. Validate Existing Time Features

In [ ]:
time_cols = ['hour','day_of_week','month','is_weekend','time_bin','hour_sin','hour_cos']
existing  = [c for c in time_cols if c in df.columns]
print(f"Pre-engineered time features present: {existing}\n")
for col in existing:
    print(f"  {col:20s}: {df[col].isnull().sum()} nulls | {df[col].nunique()} unique values")

# Recalculate sin/cos to guarantee correctness
df['hour'] = pd.to_numeric(df['hour'], errors='coerce').fillna(-1).astype(int)
h = df['hour'].clip(lower=0)
df['hour_sin'] = np.sin(2*np.pi*h/24)
df['hour_cos'] = np.cos(2*np.pi*h/24)
print("\nhour_sin / hour_cos recalculated ✓")

## 3. Additional Features

In [ ]:
# Cyclic month & day-of-week (not in original file)
df['month']       = pd.to_numeric(df['month'], errors='coerce').fillna(1).astype(int)
df['day_of_week'] = pd.to_numeric(df['day_of_week'], errors='coerce').fillna(0).astype(int)
df['month_sin']   = np.sin(2*np.pi*df['month']/12)
df['month_cos']   = np.cos(2*np.pi*df['month']/12)
df['dow_sin']     = np.sin(2*np.pi*df['day_of_week']/7)
df['dow_cos']     = np.cos(2*np.pi*df['day_of_week']/7)

# Binary time-window flags
df['is_morning']         = ((df['hour']>=9)  & (df['hour']<12)).astype(int)
df['is_early_afternoon'] = ((df['hour']>=12) & (df['hour']<15)).astype(int)
df['is_late_afternoon']  = ((df['hour']>=15) & (df['hour']<18)).astype(int)
df['is_evening']         = ((df['hour']>=18) & (df['hour']<22)).astype(int)
df['is_night']           = ((df['hour']>=22) | (df['hour']<6)).astype(int)

# Year offset
df['vergadering_datum'] = pd.to_datetime(df['vergadering_datum'], errors='coerce')
df['year']              = df['vergadering_datum'].dt.year.fillna(2013).astype(int)
df['years_since_2010']  = (df['year']-2010).clip(lower=0)

print("Calendar features ✓")

In [ ]:
# Speech metadata
df['speech_duration_seconds'] = pd.to_numeric(df['speech_duration_seconds'], errors='coerce')
p99 = df['speech_duration_seconds'].quantile(0.99)
df['speech_duration_seconds'] = df['speech_duration_seconds'].clip(upper=p99)
df['log_speech_duration'] = np.log1p(df['speech_duration_seconds'].fillna(df['speech_duration_seconds'].median()))

df['is_voorzitter'] = df['is_voorzitter'].map({True:1,False:0,'True':1,'False':0,1:1,0:0}).fillna(0).astype(int)
df['text_chars']    = df['speech_text_clean'].str.len()
df['log_text_length'] = np.log1p(df['text_chars'])
df['long_speech_flag'] = (df['text_chars'] > df['text_chars'].quantile(0.75)).astype(int)

print(f"is_voorzitter = 1: {df['is_voorzitter'].mean()*100:.1f}% of speeches")
print(f"long_speech_flag = 1: {df['long_speech_flag'].mean()*100:.1f}% of speeches")

In [ ]:
# Speaker frequency encoding
spk_freq = df['speaker_name'].value_counts()
df['speaker_freq']     = df['speaker_name'].map(spk_freq).fillna(1)
df['log_speaker_freq'] = np.log1p(df['speaker_freq'])

# Top-10 party one-hot
top_parties = df['speaker_party'].value_counts().head(10).index.tolist()
df['speaker_party_clean'] = df['speaker_party'].where(df['speaker_party'].isin(top_parties), other='Other')
party_dummies = pd.get_dummies(df['speaker_party_clean'], prefix='party')
df = pd.concat([df, party_dummies], axis=1)

# Ordinal time_bin
TIME_BIN_ORDER = {'early_morning':0,'morning':1,'early_afternoon':2,'late_afternoon':3,'evening':4,'night':5}
df['time_bin_ordinal'] = df['time_bin'].map(TIME_BIN_ORDER).fillna(2).astype(int)

# Interactions
df['voorzitter_x_evening'] = df['is_voorzitter'] * df['is_evening']
df['voorzitter_x_night']   = df['is_voorzitter'] * df['is_night']
df['duration_x_evening']   = df['log_speech_duration'] * df['is_evening']

party_cols = [c for c in df.columns if c.startswith('party_')]
print(f"Party one-hot columns: {party_cols}")

## 4. Class Imbalance

In [ ]:
n_pass = int(df['label'].sum()); n_rej = len(df) - n_pass
ratio  = max(n_pass, n_rej) / min(n_pass, n_rej)
print(f"Passed  : {n_pass:,} ({n_pass/len(df)*100:.1f}%)")
print(f"Rejected: {n_rej:,} ({n_rej/len(df)*100:.1f}%)")
print(f"Ratio   : {ratio:.2f}:1")
if ratio > 2:
    print(f"\n⚠️  Imbalance detected — XGBoost scale_pos_weight = {n_rej/n_pass:.2f}")
    print("   → Logistic Regression: class_weight='balanced'")
    print("   → RobBERT: weighted CrossEntropyLoss")
else:
    print("\n✓  Classes roughly balanced")
meta = {"n_pass":n_pass,"n_rej":n_rej,"scale_pos_weight":round(n_rej/n_pass,4)}
with open("class_meta.json","w") as f: json.dump(meta,f)
print("Saved class_meta.json ✓")

## 5. Train / Val / Test Split

In [ ]:
ids = df['speech_id']; y = df['label'].astype(int)
ids_tr, ids_tmp, _, y_tmp = train_test_split(ids, y, test_size=0.30, random_state=42, stratify=y)
ids_val, ids_te = train_test_split(ids_tmp, test_size=0.50, random_state=42, stratify=y_tmp)
df['split'] = 'train'
df.loc[df['speech_id'].isin(ids_val.values),'split'] = 'val'
df.loc[df['speech_id'].isin(ids_te.values),'split']  = 'test'
for split, grp in df.groupby('split'):
    print(f"  {split:6s}: {len(grp):6,} rows  |  {grp['label'].mean()*100:.1f}% passed")

## 6. Feature Correlation with Target

In [ ]:
STRUCTURED_COLS = [
    'hour','hour_sin','hour_cos','time_bin_ordinal',
    'is_morning','is_early_afternoon','is_late_afternoon','is_evening','is_night',
    'month_sin','month_cos','dow_sin','dow_cos','is_weekend','years_since_2010',
    'is_voorzitter','log_speech_duration','log_text_length','long_speech_flag',
    'log_speaker_freq','voorzitter_x_evening','voorzitter_x_night','duration_x_evening',
]
party_cols = [c for c in df.columns if c.startswith('party_')]
ALL_FEAT = [c for c in STRUCTURED_COLS + party_cols if c in df.columns]

corr = df[ALL_FEAT+['label']].corr()['label'].drop('label').sort_values()
fig, ax = plt.subplots(figsize=(6,10))
cols_c = ['#C0392B' if v<0 else '#27AE60' for v in corr.values]
ax.barh(corr.index, corr.values, color=cols_c, edgecolor='white', alpha=0.85)
ax.axvline(0, color='black', lw=0.7)
ax.set_title("Correlation with 'label' (motion_passed)\ngreen=→passing  red=→rejection", fontweight='bold')
ax.set_xlabel("Pearson r")
plt.tight_layout(); plt.show()

print("\nTop 5 → PASSING:"); print(corr.tail(5).to_string())
print("\nTop 5 → REJECTION:"); print(corr.head(5).to_string())

with open("feature_cols.json","w") as f:
    json.dump({"structured":STRUCTURED_COLS,"party":party_cols},f)
print("\nSaved feature_cols.json ✓")

## 7. Save

In [ ]:
df.to_csv("speeches_features.csv", index=False)
print(f"Saved {len(df):,} rows → speeches_features.csv")